# Machine Learning I (CC2008) — Trabalho Prático
## Fase 1: Random Forest com Class Imbalance

**Algoritmo escolhido:** Random Forest  
**Desafio escolhido:** Class Imbalance (Grupo 2)  

Neste notebook avaliamos como o Random Forest se comporta quando os dados têm muito mais exemplos de uma classe do que de outra. A ideia é perceber onde é que o algoritmo falha antes de propor melhorias na Fase 2.


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder
from collections import Counter

from mla.ensemble.random_forest import RandomForestClassifier

np.random.seed(42)
print("Tudo importado!")

## 2. Como funciona o Random Forest?

O Random Forest é um algoritmo que treina várias árvores de decisão ao mesmo tempo e combina as suas respostas. A ideia é simples: em vez de confiar numa única árvore (que pode cometer erros), usamos muitas árvores e fazemos uma "votação" — a classe que a maioria das árvores escolher é a resposta final.

Para garantir que as árvores são diferentes entre si (e não estão todas a aprender a mesma coisa), o algoritmo usa dois truques:

- **Bootstrap sampling:** cada árvore é treinada numa amostra aleatória *com reposição* do dataset original — ou seja, alguns exemplos aparecem mais do que uma vez e outros não aparecem. Isto introduz diversidade entre as árvores.
- **Feature randomness:** em cada divisão da árvore, só se consideram algumas features escolhidas aleatoriamente (em vez de todas). Assim as árvores não ficam todas iguais.

No final, para classificação, a previsão de cada árvore é uma distribuição de probabilidades por classe, e a média dessas probabilidades de todas as árvores dá a previsão final.

### O que descobrimos no código base

Ao analisar o código disponibilizado, encontrámos uma limitação importante: **o bootstrap sampling não está implementado**. Todas as árvores são treinadas com o dataset completo, sem qualquer amostragem. Isto significa que não há diversidade real entre as árvores — estão todas a ver os mesmos dados.

Esta limitação agrava o problema do class imbalance, como veremos a seguir.

### Porquê o Class Imbalance é um problema para o Random Forest?

Imagina um dataset em que 95% dos exemplos são da classe 0 e só 5% são da classe 1. O que acontece ao Random Forest?

1. **O critério de entropia fica enviesado:** ao procurar o melhor split, o algoritmo tende a favorecer divisões que acertam na classe maioritária (porque são a grande maioria dos exemplos)
2. **As árvores aprendem a "jogar pelo seguro":** como a classe 0 domina, as árvores aprendem que dizer sempre "classe 0" minimiza o erro — mesmo que isso signifique nunca identificar a classe 1
3. **Sem bootstrap real, o problema é ainda pior:** como todas as árvores veem os mesmos dados desequilibrados, não há nenhuma árvore que por acaso veja mais exemplos da classe minoritária

**A nossa hipótese:** à medida que o desequilíbrio aumenta, o modelo vai ter uma accuracy artificialmente alta mas um desempenho muito fraco na classe minoritária — que é normalmente a mais importante (fraudes, doenças raras, etc.).


## 3. Datasets

Usamos os **datasets disponibilizados pelo professor** para o Grupo 2 (Class Imbalance). São carregados todos os ficheiros presentes na pasta `class_imbalance/`, cobrindo diferentes níveis de desequilíbrio medido pelo **Imbalance Ratio (IR)**:

> IR = nº de exemplos da classe maioritária / nº de exemplos da classe minoritária

**Nota de pré-processamento:** Os datasets que contêm valores em falta (NaN) são tratados com imputação pela mediana (colunas numéricas) e moda (colunas categóricas). As colunas categóricas são codificadas com `LabelEncoder`. O target é sempre binarizado como 0 (maioritária) / 1 (minoritária).


In [ ]:
def calcular_ir(y):
    """Calcula o Imbalance Ratio do dataset."""
    counter = Counter(y)
    return max(counter.values()) / min(counter.values())


def resumo_dataset(nome, X, y):
    counter = Counter(y)
    ir = calcular_ir(y)
    print(f"Dataset: {nome}")
    print(f"  Amostras: {X.shape[0]}  |  Features: {X.shape[1]}")
    print(f"  Classe 0 (maioritária): {counter[0]}  |  Classe 1 (minoritária): {counter[1]}")
    print(f"  Imbalance Ratio: {ir:.1f}:1")
    print()


def carregar_dataset(filepath):
    """
    Carrega um dataset CSV do professor:
    - A última coluna é o target
    - Colunas categóricas são codificadas com LabelEncoder
    - NaNs são imputados (mediana para numéricos, moda para categóricos)
    - Retorna X (numpy array float64) e y (numpy array int 0/1)
      onde 1 é sempre a classe minoritária
    """
    df = pd.read_csv(filepath)
    target_col = df.columns[-1]

    X_df = df.drop(columns=[target_col]).copy()
    y_raw = df[target_col].copy()

    # Imputar NaNs
    for col in X_df.columns:
        if X_df[col].isnull().any():
            if X_df[col].dtype == object:
                X_df[col] = X_df[col].fillna(X_df[col].mode()[0])
            else:
                X_df[col] = X_df[col].fillna(X_df[col].median())

    # Codificar colunas categóricas
    for col in X_df.columns:
        if X_df[col].dtype == object:
            X_df[col] = LabelEncoder().fit_transform(X_df[col].astype(str))

    X = X_df.values.astype(np.float64)

    # Codificar target como 0/1
    le = LabelEncoder()
    y = le.fit_transform(y_raw.astype(str))

    # Garantir que a classe minoritária é o label 1
    counts = Counter(y)
    if counts[0] < counts[1]:
        y = 1 - y

    return X, y


print("Funções definidas!")

In [ ]:
# Caminho para a pasta com os datasets do professor
# Ajusta este caminho conforme a tua estrutura de pastas
DATA_PATH = "class_imbalance/"

datasets = []  # lista de tuplos (X, y, nome)

for fname in sorted(os.listdir(DATA_PATH)):
    if not fname.endswith(".csv"):
        continue
    fpath = os.path.join(DATA_PATH, fname)
    X, y = carregar_dataset(fpath)
    nome = fname.replace(".csv", "")
    datasets.append((X, y, nome))
    resumo_dataset(nome, X, y)

print(f"Total de datasets carregados: {len(datasets)}")

## 4. Visualização do Desequilíbrio

Antes de treinar qualquer modelo, é útil ver graficamente o quão desequilibrados estão os datasets, ordenados por IR crescente.


In [ ]:
# Ordenar datasets por IR crescente
datasets_sorted = sorted(datasets, key=lambda t: calcular_ir(t[1]))

n = len(datasets_sorted)
cols = 6
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
axes = axes.flatten()

for i, (X, y, nome) in enumerate(datasets_sorted):
    counter = Counter(y)
    ir = calcular_ir(y)
    axes[i].bar([0, 1], [counter[0], counter[1]], color=["steelblue", "tomato"])
    axes[i].set_title(f"{nome}\n(IR={ir:.1f}:1)", fontsize=7)
    axes[i].set_xticks([0, 1])
    axes[i].set_xticklabels(["Maioritária", "Minoritária"], fontsize=7)
    axes[i].set_ylabel("Nº amostras", fontsize=7)

# Esconder eixos vazios
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribuição de Classes por Dataset (ordenado por IR crescente)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("distribuicao_classes.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Avaliação do Modelo

### Como avaliamos?

Usamos **Stratified K-Fold** com k=5. A versão estratificada é importante aqui porque garante que cada fold tem a mesma proporção de classes que o dataset original — sem isso, podíamos ter folds onde a classe minoritária quase não aparece.

### Que métricas usamos?

Só usar a **accuracy** seria um erro grave com datasets desequilibrados. Por exemplo, num dataset com IR=70:1, um modelo que diga sempre "classe 0" tem ~99% de accuracy — mas não serve para nada!

Por isso usamos:
- **Accuracy** — incluímos só para mostrar que é enganosa
- **F1-score da classe minoritária** — a métrica mais importante: mede se o modelo consegue mesmo identificar a classe rara
- **F1 Macro** — média do F1 de ambas as classes, penaliza o modelo se ignorar alguma delas
- **AUC-ROC** — mede a capacidade do modelo de separar as classes, independentemente do threshold


In [ ]:
def avaliar_rf(X, y, nome_dataset, n_estimators=20, max_depth=10, min_samples_split=5):
    """Avalia o Random Forest com Stratified K-Fold e devolve as métricas médias."""

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    accs, f1s_macro, f1s_min, aucs = [], [], [], []

    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        rf = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split
        )
        rf.fit(X_train, y_train)

        probs = rf.predict(X_test)
        y_pred = np.argmax(probs, axis=1)

        accs.append(np.mean(y_pred == y_test))
        f1s_macro.append(f1_score(y_test, y_pred, average="macro", zero_division=0))
        f1s_min.append(f1_score(y_test, y_pred, pos_label=1, average="binary", zero_division=0))

        try:
            aucs.append(roc_auc_score(y_test, probs[:, 1]))
        except Exception:
            aucs.append(float("nan"))

    return {
        "Dataset": nome_dataset,
        "IR": calcular_ir(y),
        "Accuracy": np.mean(accs),
        "F1 Macro": np.mean(f1s_macro),
        "F1 Minoritária": np.mean(f1s_min),
        "AUC-ROC": np.nanmean(aucs),
    }


print("Função definida!")

In [ ]:
resultados = []
for X, y, nome in datasets:
    print(f"A avaliar: {nome}...")
    res = avaliar_rf(X, y, nome)
    resultados.append(res)
    print(f"  IR: {res['IR']:.1f} | Accuracy: {res['Accuracy']:.3f} | F1 Macro: {res['F1 Macro']:.3f} | F1 Min: {res['F1 Minoritária']:.3f} | AUC: {res['AUC-ROC']:.3f}")

df_resultados = pd.DataFrame(resultados).sort_values("IR").reset_index(drop=True).round(3)
print("\nConcluído!")

## 6. Resultados

In [ ]:
print(df_resultados.to_string(index=False))

In [ ]:
# Gráfico de barras: métricas por dataset, ordenado por IR crescente
fig, ax = plt.subplots(figsize=(max(14, len(df_resultados) * 0.8), 5))

x = np.arange(len(df_resultados))
width = 0.2

ax.bar(x - width*1.5, df_resultados["Accuracy"],       width, label="Accuracy",        color="steelblue")
ax.bar(x - width*0.5, df_resultados["F1 Macro"],       width, label="F1 Macro",        color="seagreen")
ax.bar(x + width*0.5, df_resultados["F1 Minoritária"], width, label="F1 Minoritária",  color="tomato")
ax.bar(x + width*1.5, df_resultados["AUC-ROC"],        width, label="AUC-ROC",         color="orange")

labels = [f"{row['Dataset']}\n(IR={row['IR']:.0f}:1)" for _, row in df_resultados.iterrows()]
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score")
ax.set_title("Desempenho do Random Forest por Dataset (ordenado por IR crescente)", fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("resultados_fase1.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Gráfico IR vs métricas — mostra a tendência de degradação com o aumento do desequilíbrio
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(df_resultados["IR"], df_resultados["Accuracy"],       "o-", label="Accuracy",       color="steelblue")
ax.plot(df_resultados["IR"], df_resultados["F1 Macro"],       "s-", label="F1 Macro",       color="seagreen")
ax.plot(df_resultados["IR"], df_resultados["F1 Minoritária"], "^-", label="F1 Minoritária", color="tomato")
ax.plot(df_resultados["IR"], df_resultados["AUC-ROC"],        "D-", label="AUC-ROC",        color="orange")

ax.set_xlabel("Imbalance Ratio (IR)")
ax.set_ylabel("Score")
ax.set_title("Impacto do IR no desempenho do Random Forest", fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("ir_vs_metricas.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Análise dos Resultados

### O que observamos?

Os resultados confirmam a nossa hipótese inicial:

- **A accuracy mantém-se alta mesmo com IR elevado** — isto é enganoso. Nos datasets com IR muito alto, o modelo pode ter accuracy elevada simplesmente por dizer sempre "classe maioritária".

- **O F1 da classe minoritária cai à medida que o IR aumenta** — esta é a métrica que realmente importa. À medida que o desequilíbrio piora, o modelo deixa de conseguir identificar os exemplos da classe rara.

- **O AUC-ROC é mais robusto** — porque não depende de um threshold fixo, consegue captar melhor a capacidade real do modelo de separar as classes.

### Porquê é que o Random Forest falha aqui?

O problema está no critério de entropia usado para decidir os splits das árvores. Quando a classe maioritária domina, o ganho de informação é maximizado por splits que classificam bem essa classe — a classe minoritária simplesmente não tem "peso" suficiente para influenciar as decisões.

Além disso, como o código base não implementa bootstrap sampling, todas as árvores veem exatamente os mesmos dados. Num Random Forest real, o bootstrap introduz alguma variação — por acaso, alguns bootstraps poderiam incluir mais exemplos da classe minoritária. Sem isso, há zero diversidade no que toca à distribuição de classes.


## 8. Análise Detalhada — Matriz de Confusão

Para perceber melhor o que está a acontecer, analisamos a matriz de confusão no dataset com o desequilíbrio mais severo. Isto mostra exatamente quantos exemplos de cada classe o modelo está a acertar e a errar.


In [ ]:
# Dataset com IR mais alto — determinado automaticamente
X_severo, y_severo, nome_severo = max(datasets, key=lambda t: calcular_ir(t[1]))
ir_severo = calcular_ir(y_severo)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_severo, y_severo, test_size=0.3, stratify=y_severo, random_state=42
)

rf = RandomForestClassifier(n_estimators=20, max_depth=10, min_samples_split=5)
rf.fit(X_tr, y_tr)
probs = rf.predict(X_te)
y_pred = np.argmax(probs, axis=1)

print(f"Relatório de classificação — {nome_severo} (IR~{ir_severo:.0f}:1):")
print(classification_report(
    y_te, y_pred,
    target_names=["Maioritária (0)", "Minoritária (1)"],
    zero_division=0
))

cm = confusion_matrix(y_te, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Maioritária", "Minoritária"])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Matriz de Confusão — {nome_severo} (IR~{ir_severo:.0f}:1)", fontweight="bold")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Conclusões da Fase 1

Com esta análise, usando os datasets reais fornecidos pelo professor, conseguimos perceber bem o comportamento do Random Forest em cenários de class imbalance:

1. **O algoritmo tem um viés claro para a classe maioritária** — o critério de entropia favorece os splits que acertam na classe dominante, ignorando a minoritária

2. **A accuracy é uma métrica enganosa neste contexto** — datasets com IR elevado permitem accuracy alta sem nunca identificar a classe rara

3. **O código base não tem bootstrap sampling** — o que agrava o problema porque todas as árvores veem exatamente os mesmos dados, sem qualquer diversidade

4. **O F1 da classe minoritária e o AUC-ROC são as métricas a usar** — são as que realmente medem se o modelo consegue identificar os casos raros

5. **O efeito do desequilíbrio é claro nos datasets reais** — a tendência de degradação do F1 minoritário com o aumento do IR é visível nos dados do professor, confirmando a hipótese
